### Pricing Features: Supplier-Side Impact Report (Bookings + GMV)

**Goal**
- Quantify whether suppliers who use pricing features achieve **higher bookings and GMV**.
- Describe **what feature usage looks like** (adoption rates, combinations, intensity, and where adoption is concentrated).
- Provide **PM-ready insights**: what to prioritize, which segments to target, and what adoption gaps matter.

**Strict scope**
- Supplier-side only.
- Do **not** use repeat-customer metrics.
- Use **only** the existing base audit tables in the warehouse (loaded below).

**Data sources**
- `production.supply_analytics.pricing_feature_audit_base`
- `production.supply_analytics.pricing_feature_audit_performance`

**Interpretation note**
- Results are associative unless feature adoption timestamps exist. This notebook adds adjusted comparisons (controls + matching) to reduce selection bias, but it is not a causal guarantee.

In [0]:

# Notebook setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cycler

# Enable grid and update its appearance
plt.rcParams.update({'axes.grid': True})
plt.rcParams.update({'grid.color': 'silver'})
plt.rcParams.update({'grid.linestyle': '--'})

# Set figure resolution
plt.rcParams.update({'figure.dpi': 150})

# Hide the top and right spines
plt.rcParams.update({'axes.spines.top': False})
plt.rcParams.update({'axes.spines.right': False})

# Increase font sizes
plt.rcParams.update({'font.size': 12})  # General font size
plt.rcParams.update({'axes.titlesize': 14})  # Title font size
plt.rcParams.update({'axes.labelsize': 12})  # Axis label font size

plt.rcParams.update({'axes.prop_cycle': cycler.cycler('color', ['#FF5533'])})

plt.style.use('default')
plt.rcParams.update({
    'figure.dpi': 140,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

### 1) Load base tables (tour configuration + performance)

We join configuration to performance at `tour_id`. All downstream rollups (supplier, segment, location) are derived from this merged dataset.

In [0]:

# Load configuration (feature flags, segmentation) and performance (L12M outcomes)
config_df = spark.sql("""
    SELECT *
    FROM production.supply_analytics.pricing_feature_audit_base
""").toPandas()

perf_df = spark.sql("""
    SELECT *
    FROM production.supply_analytics.pricing_feature_audit_performance
""").toPandas()

# Keep supplier-side performance columns
# IMPORTANT: repeat_customer_rate_* is intentionally excluded.
perf_keep = [
    'tour_id',
    'gmv_l12mo',
    'bookings_l12mo',
    'tickets_l12mo',
    'avg_booking_value_l12mo',
    'nr_l12mo'
]

missing_perf = [c for c in perf_keep if c not in perf_df.columns]
if missing_perf:
    print('WARNING: Missing expected performance columns:', missing_perf)

perf_df = perf_df[[c for c in perf_keep if c in perf_df.columns]].copy()

df = config_df.merge(perf_df, on='tour_id', how='left')

print('Rows (config):', len(config_df))
print('Rows (perf):  ', len(perf_df))
print('Rows (merged):', len(df))
print('Unique tours:', df['tour_id'].nunique())
print('Unique suppliers:', df['supplier_id'].nunique())

### 2) Basic cleaning + analysis population

We will:
- Convert numeric columns safely.
- Create a tour-level dataset (one row per tour).
- Create a supplier-level dataset (one row per supplier).

**Population choices (transparent and editable):**
- Primary analysis uses **tours with native pricing** (`uses_external_pricing == 0`).
- For impact comparisons we focus on tours with stable performance values and require `gmv_l12mo > 0`.

In [0]:

# Safe numeric conversion
num_cols = [
    'gmv_l12mo','bookings_l12mo','tickets_l12mo','avg_booking_value_l12mo','nr_l12mo',
    'num_individual_categories','num_group_categories','num_addons','total_price_tiers',
    'pricing_dates_covered_next_12mo','pct_dates_with_vacancy_info','cutoff_hours','sample_max_vacancies'
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Feature flags present in the base table (handle missing gracefully)
feature_flags = [
    'has_individual_pricing',
    'has_group_pricing',
    'has_addons_configured',
    'has_any_scale_pricing',
    'has_capacity_limits',
    'has_cutoff_configured',
    'has_min_participant_requirement',
    'has_max_participant_limit',
    'uses_prices_over_api',
    'uses_live_availability_and_price',
    'uses_pricing_scales_over_api',
    'uses_external_pricing'
]

for f in feature_flags:
    if f in df.columns:
        df[f] = df[f].fillna(0).astype(int)

# Primary slice: native pricing tours
if 'uses_external_pricing' in df.columns:
    df_native = df[df['uses_external_pricing'] == 0].copy()
else:
    df_native = df.copy()

print('Native pricing tours:', df_native['tour_id'].nunique())
print('Native pricing suppliers:', df_native['supplier_id'].nunique())

# Build tour-level: one row per tour
agg_dict = {f:'max' for f in feature_flags if f in df_native.columns}

# Keep segmentation fields (first non-null)
seg_fields = [c for c in ['supplier_id','segment','activity_type','tour_category','location_name'] if c in df_native.columns]
for c in seg_fields:
    agg_dict[c] = 'first'

# Keep configuration intensity fields
for c in ['num_individual_categories','num_group_categories','num_addons','total_price_tiers',
          'pricing_dates_covered_next_12mo','pct_dates_with_vacancy_info','cutoff_hours','sample_max_vacancies']:
    if c in df_native.columns:
        agg_dict[c] = 'mean'

# Keep performance
outcomes = [c for c in ['gmv_l12mo','bookings_l12mo','tickets_l12mo','avg_booking_value_l12mo','nr_l12mo'] if c in df_native.columns]
for c in outcomes:
    agg_dict[c] = 'first'

tour = df_native.groupby('tour_id').agg(agg_dict).reset_index()

# Fill missing performance values (left join)
for c in outcomes:
    tour[c] = tour[c].fillna(0)

# Tours used in impact comparisons
# (Edit this if you want to include tours with GMV=0)
tour_perf = tour[tour['gmv_l12mo'] > 0].copy()

print('Tour-level rows:', len(tour))
print('Tours w/ GMV>0:', len(tour_perf))

### 3) Define pricing models, bundles, and usage intensity

We translate raw flags into:
- **Pricing model**: Individual only, Group only, Flexible (Individual + Group), None.
- **Feature count**: how many core pricing features are configured.
- **Bundle**: a readable combination string.

In [0]:

core_features = [
    'has_individual_pricing','has_group_pricing','has_addons_configured','has_any_scale_pricing',
    'has_capacity_limits','has_cutoff_configured'
]
core_features = [c for c in core_features if c in tour.columns]

# Pricing model
if {'has_individual_pricing','has_group_pricing'}.issubset(tour.columns):
    tour['pricing_model'] = np.select(
        [
            (tour['has_individual_pricing']==1) & (tour['has_group_pricing']==1),
            (tour['has_individual_pricing']==1) & (tour['has_group_pricing']==0),
            (tour['has_individual_pricing']==0) & (tour['has_group_pricing']==1),
        ],
        ['Flexible (Individual + Group)','Individual Only','Group Only'],
        default='None'
    )
else:
    tour['pricing_model'] = 'Unknown'

# Feature count (core)
if core_features:
    tour['feature_count_core'] = tour[core_features].sum(axis=1)
else:
    tour['feature_count_core'] = 0

# Bundle label

def bundle_label(r):
    ind = int(r.get('has_individual_pricing',0))
    grp = int(r.get('has_group_pricing',0))
    add = int(r.get('has_addons_configured',0))
    scl = int(r.get('has_any_scale_pricing',0))
    cap = int(r.get('has_capacity_limits',0))
    cut = int(r.get('has_cutoff_configured',0))

    base = 'Flexible' if (ind and grp) else ('Individual' if ind else ('Group' if grp else 'None'))
    enh = []
    if add: enh.append('Addons')
    if scl: enh.append('Scale')
    if cap: enh.append('Capacity')
    if cut: enh.append('Cutoff')

    if base=='None' and not enh:
        return 'No pricing features'

    return base if not enh else base + ' + ' + '+'.join(enh)

bundle_cols = {'has_individual_pricing','has_group_pricing','has_addons_configured','has_any_scale_pricing',
               'has_capacity_limits','has_cutoff_configured'}
if bundle_cols.intersection(tour.columns):
    tour['bundle'] = tour.apply(bundle_label, axis=1)
else:
    tour['bundle'] = 'Unknown'

# Mirror for perf slice
cols_to_copy = ['pricing_model','feature_count_core','bundle']
for c in cols_to_copy:
    if c in tour.columns:
        tour_perf[c] = tour_perf.merge(tour[['tour_id',c]], on='tour_id', how='left')[c]

print(tour['pricing_model'].value_counts(dropna=False))

### 4) Supplier-level dataset (primary lens)

We aggregate tour outcomes to suppliers to answer:
- Do suppliers with higher feature adoption have **more bookings / GMV**?
- Is feature usage concentrated among a small set of large suppliers?

Supplier outcomes:
- totals: GMV, bookings
- normalized: GMV per tour, bookings per tour

Supplier usage:
- share of tours with each feature
- average core feature count

In [0]:

# Supplier-level rollup (based on tours with GMV>0 for stability)
base = tour_perf.copy()

feat_for_supplier = [
    'has_individual_pricing','has_group_pricing','has_addons_configured','has_any_scale_pricing',
    'has_capacity_limits','has_cutoff_configured',
    'uses_prices_over_api','uses_live_availability_and_price','uses_pricing_scales_over_api'
]
feat_for_supplier = [c for c in feat_for_supplier if c in base.columns]

supplier_agg = {
    'tour_id':'count',
    'gmv_l12mo':'sum',
    'bookings_l12mo':'sum',
    'tickets_l12mo':'sum',
}

for f in feat_for_supplier:
    supplier_agg[f] = 'mean'  # mean of 0/1 = share of supplier tours using feature

if 'feature_count_core' in base.columns:
    supplier_agg['feature_count_core'] = 'mean'

if 'segment' in base.columns:
    supplier_agg['segment'] = 'first'

supplier = base.groupby('supplier_id').agg(supplier_agg).reset_index()

supplier = supplier.rename(columns={
    'tour_id':'tours_active',
    'gmv_l12mo':'gmv_l12mo_total',
    'bookings_l12mo':'bookings_l12mo_total',
    'tickets_l12mo':'tickets_l12mo_total',
})

supplier['gmv_per_tour'] = supplier['gmv_l12mo_total'] / supplier['tours_active']
supplier['bookings_per_tour'] = supplier['bookings_l12mo_total'] / supplier['tours_active']

# Weighted supplier AOV (optional)
if 'avg_booking_value_l12mo' in base.columns:
    tmp = base[['supplier_id','avg_booking_value_l12mo','bookings_l12mo']].copy()
    tmp['w'] = tmp['bookings_l12mo'].clip(lower=0)
    tmp['wx'] = tmp['avg_booking_value_l12mo'] * tmp['w']
    aov = tmp.groupby('supplier_id').agg(wsum=('w','sum'), wxsum=('wx','sum')).reset_index()
    aov['aov_weighted'] = np.where(aov['wsum']>0, aov['wxsum']/aov['wsum'], np.nan)
    supplier = supplier.merge(aov[['supplier_id','aov_weighted']], on='supplier_id', how='left')

print('Suppliers in analysis:', len(supplier))
print(supplier[['tours_active','gmv_l12mo_total','bookings_l12mo_total','gmv_per_tour','bookings_per_tour']].describe())

## Executive summary tables

PM-friendly tables:
- adoption at tour level
- adoption at supplier level

In [0]:

# Adoption at tour level

def adoption_table_tour(tour_df):
    rows = []
    for f, label in [
        ('has_individual_pricing','Individual pricing'),
        ('has_group_pricing','Group pricing'),
        ('has_addons_configured','Addons'),
        ('has_any_scale_pricing','Scale pricing'),
        ('has_capacity_limits','Capacity limits'),
        ('has_cutoff_configured','Cutoff time'),
        ('uses_prices_over_api','Prices over API'),
        ('uses_live_availability_and_price','Live availability & price'),
        ('uses_pricing_scales_over_api','Scales over API')
    ]:
        if f in tour_df.columns:
            rows.append({
                'feature': label,
                'tours_using': int((tour_df[f]==1).sum()),
                'tours_total': int(len(tour_df)),
                'adoption_rate_pct': float((tour_df[f].mean()*100).round(1))
            })
    return pd.DataFrame(rows).sort_values('adoption_rate_pct', ascending=False)

label_map = {
    'has_individual_pricing':'Individual pricing',
    'has_group_pricing':'Group pricing',
    'has_addons_configured':'Addons',
    'has_any_scale_pricing':'Scale pricing',
    'has_capacity_limits':'Capacity limits',
    'has_cutoff_configured':'Cutoff time',
    'uses_prices_over_api':'Prices over API',
    'uses_live_availability_and_price':'Live availability & price',
    'uses_pricing_scales_over_api':'Scales over API'
}

exec_adoption = adoption_table_tour(tour)
exec_adoption

In [0]:

# Adoption at supplier level (share of suppliers that use feature on >=1 tour)
rows=[]
for f, label in label_map.items():
    if f in supplier.columns:
        pct = (supplier[f] > 0).mean() * 100
        rows.append({'feature': label, 'suppliers_using_pct': round(pct,1)})

supplier_adoption = pd.DataFrame(rows).sort_values('suppliers_using_pct', ascending=False)
supplier_adoption

### 5) What usage looks like (distribution + concentration)

Questions answered:
- Is adoption intensity common or concentrated?
- How much GMV sits in low vs high feature-intensity suppliers?

In [0]:

fig, ax = plt.subplots(figsize=(10,5))
ax.hist(supplier['feature_count_core'].fillna(0), bins=np.arange(0,7,0.5), edgecolor='black')
ax.set_title('Supplier pricing-feature intensity (avg core features across tours)')
ax.set_xlabel('Average number of core pricing features per supplier')
ax.set_ylabel('Number of suppliers')
plt.tight_layout()
plt.show()

In [0]:

# Intensity bands and GMV share
supplier2 = supplier.copy()

supplier2['intensity_band'] = pd.cut(
    supplier2['feature_count_core'].fillna(0),
    bins=[-0.01, 0.5, 1.5, 2.5, 6],
    labels=['0 features','~1 feature','~2 features','3+ features']
)

band = supplier2.groupby('intensity_band', observed=True).agg(
    suppliers=('supplier_id','count'),
    tours=('tours_active','sum'),
    gmv=('gmv_l12mo_total','sum'),
    bookings=('bookings_l12mo_total','sum'),
    gmv_per_tour=('gmv_per_tour','mean'),
    bookings_per_tour=('bookings_per_tour','mean'),
).reset_index()

band['gmv_share_pct'] = (band['gmv'] / band['gmv'].sum() * 100).round(1)
band['bookings_share_pct'] = (band['bookings'] / band['bookings'].sum() * 100).round(1)

band

## 6) Core question: do suppliers get more bookings and GMV when they use features?

We run:
- Tour-level with vs without comparisons
- Supplier-level users vs non-users comparisons

No repeat-customer metrics are used.

In [0]:

# Helper: compute lift for a single binary feature

def with_without_lift(df_in, feature_col, outcome_col, min_group_n=100):
    if feature_col not in df_in.columns or outcome_col not in df_in.columns:
        return None
    a = df_in[df_in[feature_col]==1][outcome_col]
    b = df_in[df_in[feature_col]==0][outcome_col]

    if len(a) < min_group_n or len(b) < min_group_n:
        return None

    mean_a = a.mean()
    mean_b = b.mean()
    lift_pct = ((mean_a/mean_b)-1)*100 if mean_b>0 else np.nan

    med_a = a.median()
    med_b = b.median()
    lift_med_pct = ((med_a/med_b)-1)*100 if med_b>0 else np.nan

    return {
        'feature': feature_col,
        'feature_name': label_map.get(feature_col, feature_col),
        'outcome': outcome_col,
        'n_with': int(len(a)),
        'n_without': int(len(b)),
        'mean_lift_pct': float(lift_pct),
        'median_lift_pct': float(lift_med_pct),
        'mean_with': float(mean_a),
        'mean_without': float(mean_b),
    }

features_eval = [
    'has_individual_pricing','has_group_pricing','has_addons_configured','has_any_scale_pricing',
    'has_capacity_limits','has_cutoff_configured',
    'uses_prices_over_api','uses_live_availability_and_price','uses_pricing_scales_over_api'
]
features_eval = [f for f in features_eval if f in tour_perf.columns]

rows=[]
for f in features_eval:
    for y in ['gmv_l12mo','bookings_l12mo']:
        r = with_without_lift(tour_perf, f, y)
        if r:
            rows.append(r)

lift_tour = pd.DataFrame(rows)

lift_tour[['feature_name','outcome','n_with','n_without','mean_lift_pct','median_lift_pct']].sort_values(['outcome','mean_lift_pct'], ascending=[True, False])

In [0]:

# Visual: tour-level mean lifts
for outcome in ['gmv_l12mo','bookings_l12mo']:
    d = lift_tour[lift_tour['outcome']==outcome].copy().sort_values('mean_lift_pct', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 0.45*len(d)+2))
    ax.barh(d['feature_name'], d['mean_lift_pct'], edgecolor='black')
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title(f'With vs without feature: mean lift in {outcome} (tour-level)')
    ax.set_xlabel('Lift (%)')

    for i, (v, n1) in enumerate(zip(d['mean_lift_pct'], d['n_with'])):
        ax.text(v + (1 if v>=0 else -1), i, f"{v:.1f}%  (with: {n1:,})",
                va='center', ha='left' if v>=0 else 'right', fontsize=9)

    plt.tight_layout()
    plt.show()

### 6.1 Supplier-level users vs non-users

A supplier is a **user** of a feature if they have the feature on at least one active tour.
We focus on per-tour outcomes to reduce the effect of supplier size.

In [0]:

rows=[]
for f in features_eval:
    if f not in supplier.columns:
        continue

    use = (supplier[f] > 0).astype(int)

    for y in ['gmv_per_tour','bookings_per_tour','gmv_l12mo_total','bookings_l12mo_total']:
        a = supplier[use==1][y]
        b = supplier[use==0][y]
        if len(a) >= 50 and len(b) >= 50:
            mean_a, mean_b = a.mean(), b.mean()
            lift = ((mean_a/mean_b)-1)*100 if mean_b>0 else np.nan
            rows.append({
                'feature': f,
                'feature_name': label_map.get(f,f),
                'outcome': y,
                'suppliers_with': int((use==1).sum()),
                'suppliers_without': int((use==0).sum()),
                'mean_lift_pct': float(lift)
            })

lift_supplier = pd.DataFrame(rows)

lift_supplier.sort_values(['outcome','mean_lift_pct'], ascending=[True, False]).head(40)

In [0]:

# Visual: supplier-level per-tour lifts
for outcome in ['gmv_per_tour','bookings_per_tour']:
    d = lift_supplier[lift_supplier['outcome']==outcome].copy().sort_values('mean_lift_pct', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 0.45*len(d)+2))
    ax.barh(d['feature_name'], d['mean_lift_pct'], edgecolor='black')
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title(f'Supplier-level association: lift in {outcome} (users vs non-users)')
    ax.set_xlabel('Lift (%)')

    for i, (v, n1) in enumerate(zip(d['mean_lift_pct'], d['suppliers_with'])):
        ax.text(v + (1 if v>=0 else -1), i, f"{v:.1f}%  (suppliers with: {n1:,})",
                va='center', ha='left' if v>=0 else 'right', fontsize=9)

    plt.tight_layout()
    plt.show()

## 7) Adjusted comparisons (controls + clustered errors)

We estimate adjusted associations using:
- $$\log(1 + GMV)$$ and $$\log(1 + bookings)$$
- Controls: `segment`, `activity_type`, `location_name` (if present), plus `tours_per_supplier`
- Clustered standard errors at `supplier_id`

In [0]:

import statsmodels.formula.api as smf

model_df = tour_perf.copy()

# Supplier size proxy
supplier_sizes = model_df.groupby('supplier_id').agg(tours_per_supplier=('tour_id','count')).reset_index()
model_df = model_df.merge(supplier_sizes, on='supplier_id', how='left')

cat_controls = [c for c in ['segment','activity_type','location_name'] if c in model_df.columns]
for c in cat_controls:
    model_df[c] = model_df[c].astype('string').fillna('Unknown')

model_df['log_gmv'] = np.log1p(model_df['gmv_l12mo'])
model_df['log_bookings'] = np.log1p(model_df['bookings_l12mo'])

X_feats = features_eval.copy()  # includes core + API flags

controls_formula = ' + '.join([f'C({c})' for c in cat_controls] + ['tours_per_supplier'])
X_formula = ' + '.join(X_feats)

formula_gmv = f"log_gmv ~ {X_formula} + {controls_formula}" if controls_formula else f"log_gmv ~ {X_formula}"
formula_bk  = f"log_bookings ~ {X_formula} + {controls_formula}" if controls_formula else f"log_bookings ~ {X_formula}"

print('GMV model formula:', formula_gmv)
print('Bookings model formula:', formula_bk)

fit_gmv = smf.ols(formula_gmv, data=model_df).fit(cov_type='cluster', cov_kwds={'groups': model_df['supplier_id']})
fit_bk  = smf.ols(formula_bk,  data=model_df).fit(cov_type='cluster', cov_kwds={'groups': model_df['supplier_id']})


def coef_table(fit, feats):
    out=[]
    for f in feats:
        if f in fit.params.index:
            beta = fit.params[f]
            se = fit.bse[f]
            p = fit.pvalues[f]
            pct = (np.expm1(beta) * 100)
            out.append({
                'feature': f,
                'feature_name': label_map.get(f,f),
                'lift_pct_adj': float(pct),
                'p_value': float(p),
                'beta_log': float(beta),
                'se_log': float(se)
            })
    return pd.DataFrame(out).sort_values('lift_pct_adj', ascending=False)

adj_gmv = coef_table(fit_gmv, X_feats)
adj_bk  = coef_table(fit_bk,  X_feats)

adj_gmv

In [0]:

adj_bk

In [0]:

# Visual adjusted lifts

def plot_adj(adj_df, title):
    d = adj_df.sort_values('lift_pct_adj', ascending=True)
    fig, ax = plt.subplots(figsize=(10, 0.45*len(d)+2))
    ax.barh(d['feature_name'], d['lift_pct_adj'], edgecolor='black')
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title(title)
    ax.set_xlabel('Adjusted lift (%) from log model')

    for i, (v, p) in enumerate(zip(d['lift_pct_adj'], d['p_value'])):
        ax.text(v + (1 if v>=0 else -1), i, f"{v:.1f}%  (p={p:.3g})",
                va='center', ha='left' if v>=0 else 'right', fontsize=9)

    plt.tight_layout()
    plt.show()

plot_adj(adj_gmv, 'Adjusted association with GMV (tour-level, with controls)')
plot_adj(adj_bk,  'Adjusted association with bookings (tour-level, with controls)')

## 8) Propensity-score matching (optional robustness)

For each selected feature:
- model adoption probability from controls
- nearest-neighbor match
- compute ATT on log outcomes

This is optional but useful if you want a second bias-reduction view.

In [0]:

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score


def psm_att(model_df, feature, outcome_log, cat_controls, size_col='tours_per_supplier', max_rows=200000, random_state=42):
    d = model_df[[feature, outcome_log, 'supplier_id'] + cat_controls + [size_col]].dropna().copy()

    # Optional downsample for speed
    if len(d) > max_rows:
        d = d.sample(max_rows, random_state=random_state)

    y = d[feature].astype(int).values
    X = d[cat_controls + [size_col]]

    pre = ColumnTransformer(
        transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), cat_controls)],
        remainder='passthrough'
    )
    clf = LogisticRegression(max_iter=200)
    pipe = Pipeline([('pre', pre), ('clf', clf)])
    pipe.fit(X, y)

    p = pipe.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan

    d['propensity'] = p
    d['treated'] = y

    treated = d[d['treated'] == 1].copy()
    control = d[d['treated'] == 0].copy().sort_values('propensity')

    if len(treated) == 0 or len(control) == 0:
        return None

    ctrl_p = control['propensity'].values
    ctrl_out = control[outcome_log].values

    idx = np.searchsorted(ctrl_p, treated['propensity'].values)
    idx = np.clip(idx, 1, len(ctrl_p) - 1)
    left = idx - 1
    right = idx

    tprop = treated['propensity'].values
    choose_right = (np.abs(ctrl_p[right] - tprop) < np.abs(ctrl_p[left] - tprop))
    match_idx = np.where(choose_right, right, left)

    att = (treated[outcome_log].values - ctrl_out[match_idx]).mean()
    att_pct = np.expm1(att) * 100

    return {
        'feature': feature,
        'feature_name': label_map.get(feature, feature),
        'outcome': outcome_log,
        'n_treated': int(len(treated)),
        'n_control_pool': int(len(control)),
        'psm_att_pct': float(att_pct),
        'propensity_auc': float(auc)
    }

key_psm_features = [
    'has_addons_configured',
    'has_any_scale_pricing',
    'has_capacity_limits',
    'has_cutoff_configured',
    'uses_live_availability_and_price'
]
key_psm_features = [f for f in key_psm_features if f in model_df.columns]

psm_rows=[]
for f in key_psm_features:
    r1 = psm_att(model_df, f, 'log_gmv', cat_controls)
    r2 = psm_att(model_df, f, 'log_bookings', cat_controls)
    if r1: psm_rows.append(r1)
    if r2: psm_rows.append(r2)

psm = pd.DataFrame(psm_rows)
psm

## 9) Configuration patterns (bundles) and outcomes

This provides a PM-friendly ladder view:
- which bundles are common
- which bundles have higher GMV/bookings per tour

We apply a minimum tour count threshold to avoid over-interpreting tiny groups.

In [0]:

bundle_perf = tour_perf.groupby('bundle').agg(
    tours=('tour_id','count'),
    suppliers=('supplier_id','nunique'),
    gmv_total=('gmv_l12mo','sum'),
    bookings_total=('bookings_l12mo','sum'),
    gmv_per_tour=('gmv_l12mo','mean'),
    bookings_per_tour=('bookings_l12mo','mean'),
).reset_index()

bundle_perf = bundle_perf[bundle_perf['tours'] >= 200].sort_values('gmv_per_tour', ascending=False)

bundle_perf

In [0]:

d = bundle_perf.sort_values('gmv_per_tour', ascending=True).tail(12)
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(d['bundle'], d['gmv_per_tour'], edgecolor='black')
ax.set_title('Top bundles by GMV per tour (native pricing tours)')
ax.set_xlabel('Average GMV per tour (L12M)')
for i, (v, n) in enumerate(zip(d['gmv_per_tour'], d['tours'])):
    ax.text(v, i, f"  €{v:,.0f}  ({n:,} tours)", va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 10) Where to focus (segment x feature): adoption gap + adjusted lift

We produce a ranked table:
- adoption rate by segment
- adjusted lift (overall)
- opportunity score proportional to segment GMV scale and adoption gap

This is a prioritization tool (not a claim of incremental GMV without experimentation).

In [0]:

if 'segment' in tour_perf.columns:
    seg_scale = tour_perf.groupby('segment').agg(
        tours=('tour_id','count'),
        suppliers=('supplier_id','nunique'),
        gmv_total=('gmv_l12mo','sum'),
        bookings_total=('bookings_l12mo','sum')
    ).reset_index()

    seg_adopt = tour_perf.groupby('segment')[features_eval].mean().reset_index()
    seg_adopt = seg_adopt.melt(id_vars='segment', var_name='feature', value_name='adoption_rate')
    seg_adopt['adoption_rate_pct'] = (seg_adopt['adoption_rate']*100).round(1)
    seg_adopt['feature_name'] = seg_adopt['feature'].map(label_map)

    adj_gmv_map = dict(zip(adj_gmv['feature'], adj_gmv['lift_pct_adj']))
    adj_bk_map = dict(zip(adj_bk['feature'], adj_bk['lift_pct_adj']))

    seg_adopt['adj_lift_gmv_pct'] = seg_adopt['feature'].map(adj_gmv_map)
    seg_adopt['adj_lift_bookings_pct'] = seg_adopt['feature'].map(adj_bk_map)

    seg_adopt = seg_adopt.merge(seg_scale, on='segment', how='left')

    seg_adopt['opportunity_score'] = (
        seg_adopt['gmv_total'].fillna(0) *
        (1 - seg_adopt['adoption_rate'].fillna(0)) *
        (seg_adopt['adj_lift_gmv_pct'].clip(lower=0).fillna(0) / 100)
    )

    action_table = seg_adopt.sort_values('opportunity_score', ascending=False)

    action_table[['segment','feature_name','adoption_rate_pct','adj_lift_gmv_pct','adj_lift_bookings_pct','tours','suppliers','gmv_total','opportunity_score']].head(25)
else:
    print('No segment column available; skipping.')

In [0]:

# Overall prioritization: adoption vs adjusted GMV lift

overall = pd.DataFrame({
    'feature': features_eval,
    'adoption_rate': [tour_perf[f].mean() for f in features_eval]
})

overall['feature_name'] = overall['feature'].map(label_map)
overall['adoption_rate_pct'] = overall['adoption_rate'] * 100
overall['adj_lift_gmv_pct'] = overall['feature'].map(dict(zip(adj_gmv['feature'], adj_gmv['lift_pct_adj'])))

d = overall.dropna(subset=['adj_lift_gmv_pct']).copy()

fig, ax = plt.subplots(figsize=(10,6))
ax.scatter(d['adoption_rate_pct'], d['adj_lift_gmv_pct'], s=220, edgecolor='black', alpha=0.85)
for _, r in d.iterrows():
    ax.annotate(r['feature_name'], (r['adoption_rate_pct'], r['adj_lift_gmv_pct']), xytext=(6,6), textcoords='offset points', fontsize=9)

ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('Adoption rate (% tours using feature)')
ax.set_ylabel('Adjusted GMV lift (%)')
ax.set_title('Prioritization view: low adoption + high adjusted lift')
plt.tight_layout()
plt.show()

## 11) PM-ready narrative output

This prints a concise summary you can paste into a doc.

In [0]:

summary_lines = []
summary_lines.append('SUPPLIER-SIDE PRICING FEATURES: PM SUMMARY')
summary_lines.append('')
summary_lines.append(f"Native-pricing tours analyzed (GMV>0): {tour_perf['tour_id'].nunique():,}")
summary_lines.append(f"Suppliers analyzed: {supplier['supplier_id'].nunique():,}")
summary_lines.append('')

# Top adjusted GMV features
if len(adj_gmv) > 0:
    top_adj = adj_gmv.sort_values('lift_pct_adj', ascending=False).head(5)
    summary_lines.append('Top features by adjusted GMV association (controls applied):')
    for _, r in top_adj.iterrows():
        summary_lines.append(f"- {r['feature_name']}: {r['lift_pct_adj']:.1f}% (p={r['p_value']:.3g})")
    summary_lines.append('')

# Top segment opportunities
if 'segment' in tour_perf.columns:
    top_opps = action_table.head(8)
    summary_lines.append('Where to focus (segment x feature), ranked by opportunity score:')
    for _, r in top_opps.iterrows():
        summary_lines.append(
            f"- {r['segment']} | {r['feature_name']}: adoption {r['adoption_rate_pct']:.1f}%, adj GMV lift {r['adj_lift_gmv_pct']:.1f}%"
        )
    summary_lines.append('')

# Bundle ladder
if len(bundle_perf) > 0:
    summary_lines.append('Best-performing configuration bundles (min sample size applied):')
    for _, r in bundle_perf.head(8).iterrows():
        summary_lines.append(
            f"- {r['bundle']}: GMV/tour €{r['gmv_per_tour']:,.0f}, bookings/tour {r['bookings_per_tour']:.1f} (tours {r['tours']:,})"
        )

print('
'.join(summary_lines))

## 12) Notes and next steps

**How to interpret impact**
- Raw comparisons show differences, but can reflect selection.
- Adjusted comparisons reduce obvious confounding but are not fully causal.

**Recommended next step for causality**
- Add feature adoption timestamps and run event studies / difference-in-differences.